# MS3 — Step 3: AlphaFold3 Benchmark vs. Crystal Structures

Benchmarks AF3-predicted HLA–peptide complexes against 4 independent, high-resolution
crystal structures (chosen for independence from the Step 2 IEDB epitope list, so this
is a genuine benchmark, not circular validation). Produces TM-score/RMSD, RCSB metadata,
and a NetMHCpan/NetMHCIIpan structural-vs-sequence concordance table.

**Benchmark set reduced from 5 to 4 (see Part A)**: a planned HLA-B*07:02/RPRGEVRFL
pair (B3) was dropped after no crystal structure of that peptide bound to any HLA
allele could be confirmed to exist on RCSB, PDBe, or PDBj.

**Compute stack:** AlphaFold3 server (free, manual web submission — no public batch API
exists) + NetMHCpan 4.1/NetMHCIIpan 4.0 (free academic license, run locally on your own
machine, results uploaded here) + TMscore/BioPython (structural comparison).

**Manual step required:** The free AlphaFold3 server (alphafoldserver.com) has no
programmatic batch API — Google's hosted server only accepts submissions through the
browser. This notebook generates every JSON input file and a tracking manifest so the
manual submission step is as low-friction as possible: submit each file, download the
result, drop it in the expected folder, and the rest of the pipeline runs automatically.

**Workflow:**
1. Generate 5 benchmark pairs + AF3 JSON inputs (Part A)
2. **[MANUAL]** Submit each JSON to alphafoldserver.com, download results into `af3_results/` (instructions printed below)
3. Ingest AF3 results, benchmark vs. crystal structures (Part B)
4. Pull RCSB metadata (Part C)
5. Run NetMHCpan locally, upload results here for concordance (Part D)
6. Export manuscript table (Part E)

## Setup

In [ ]:
# Install dependencies (Colab-safe)
!pip install -q biopython requests pandas 2>/dev/null || pip install -q --break-system-packages biopython requests pandas

import json, os, time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import requests
import pandas as pd

# Working directories
BASE_DIR = Path("/content/ms3_step3")
FASTA_DIR = BASE_DIR / "af3_inputs"
RESULTS_DIR = BASE_DIR / "af3_results"      # <- drop your downloaded AF3 results here
PDB_DIR = BASE_DIR / "crystal_pdbs"
TABLES_DIR = BASE_DIR / "tables"

for d in (FASTA_DIR, RESULTS_DIR, PDB_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {BASE_DIR}")
print(f"Drop downloaded AF3 result folders into: {RESULTS_DIR}")

## Part A — Benchmark Pairs and AF3 JSON Generation

Four independent benchmark complexes (not drawn from the Step 2 IEDB epitope list):
HLA-A\*02:01 ×2, HLA-B\*07:02 ×1, HLA-DRB1\*01:01 ×1.

**Correction applied**: B5 benchmarks against PDB 1DLH (HLA-DR1/HA 306-318). An earlier
draft of this cell referenced 1KLU for the same complex; 1DLH is the correct accession
and is what the executed pipeline actually benchmarked against (confirmed directly from
Table_S0_af3_benchmark.csv: TM-score 0.9902, RMSD 0.433 Å, 2.8 Å resolution, 1994
deposition). This cell now matches that executed result.

**Further correction applied**: B2 and B4 (and the now-dropped B3) were found to be
mis-referenced against the wrong crystal structures in an earlier draft of this cell.
Direct inspection of the predicted `.cif` files confirmed the AF3 structures themselves
are correct (right HLA heavy-chain sequence, right peptide in chain C) — only the
external crystal-structure accessions used for TM-score/RMSD comparison were wrong:

- B2 (NLVPMVATV/HLA-A\*02:01): corrected 1T1W → 3GSO. 1T1W is a real structure but of
  HLA-A2 with an HIV-1 Gag SL9 variant peptide (SLYNTVATL), unrelated to this complex.
- B4 (RPHERNGFTVL/HLA-B\*07:02): corrected 1W0V → 3VCL. 1W0V is a real structure but of
  HLA-B*2705 with an unrelated 9-residue self-peptide, not this complex.
- B3 (RPRGEVRFL/HLA-B\*07:02): dropped. The previously cited 3MGT is a real structure but
  of HLA-A*02:01 with an H5N1 influenza HA peptide, unrelated to either this allele or
  peptide. No genuine RPRGEVRFL/HLA structure could be found to replace it.

TM-score and RMSD for B2 and B4 have been recomputed against the corrected reference
structures (3GSO, 3VCL): B2 TM-score 0.9922, RMSD 0.535 Å; B4 TM-score 0.9629, RMSD
1.172 Å. The recomputation used two independent superposition methods in close
agreement — PyMOL CEAlign (window size 8, matching the method used for B1/B5; RMSD
0.535 Å and 1.172 Å respectively) and TM-align via the `tmtools` package (RMSD 0.543
Å and 1.195 Å respectively) — and both reference coordinate files were confirmed by
direct chain-sequence inspection to contain the correct heavy-chain/peptide complex
(3GSO: chain A = HLA-A heavy chain, chain P = NLVPMVATV; 3VCL: chain A = HLA-B7 heavy
chain, chain C = RPHERNGFTVL) before alignment. `Table_S0_af3_benchmark.csv` now carries
these recomputed values rather than the earlier `PENDING_RECOMPUTE` placeholders.

In [ ]:
@dataclass
class BenchmarkPair:
    pair_id: str
    hla_allele: str
    peptide: str
    pdb_id: str          # RCSB accession for benchmarking
    chain_hla: str        # chain letter for HLA chain in the reference PDB
    chain_pep: str        # chain letter for peptide in the reference PDB
    note: str = ""

BENCHMARK_PAIRS = [
    BenchmarkPair("B1", "HLA-A*02:01", "GILGFVFTL", "1OGA", "A", "C",
                  "Canonical A*02:01 benchmark (Influenza A M1 58-66)"),
    BenchmarkPair("B2", "HLA-A*02:01", "NLVPMVATV", "3GSO", "A", "C",
                  "Second independent A*02:01 structure (HCMV pp65 495-503). "
                  "Corrected from an earlier draft that cited 1T1W for this pair: "
                  "1T1W is a real PDB entry but of HLA-A2 with an HIV-1 Gag SL9 "
                  "variant (SLYNTVATL), not this NLVPMVATV/CMV pp65 complex. 3GSO "
                  "(Gras et al., J Immunol 2009;183:430-437) is the verified "
                  "NLVPMVATV/HLA-A*02:01 structure."),
    BenchmarkPair("B4", "HLA-B*07:02", "RPHERNGFTVL", "3VCL", "A", "C",
                  "11-mer, tests groove accommodation of longer peptide (HCMV pp65). "
                  "Corrected from an earlier draft that cited 1W0V for this pair: "
                  "1W0V is a real PDB entry but of HLA-B*2705 with an unrelated "
                  "9-residue self-peptide, not this complex. 3VCL (Brennan et al., "
                  "J Immunol 2012;188:2742-2748) is the verified RPHERNGFTVL/HLA-B7 "
                  "structure; note this entry's allele is annotated generically as "
                  "HLA-B7 rather than the fully resolved B*07:02 subtype."),
    BenchmarkPair("B5", "HLA-DRB1*01:01", "PKYVKQNTLKLAT", "1DLH", "A", "C",
                  "HLA-DR1/HA, standard class II benchmark (Influenza HA 307-319)"),
]

# Note: a planned B3 benchmark pair (HLA-B*07:02 / RPRGEVRFL, originally
# mis-cited against PDB 3MGT, which is actually an HLA-A*02:01/H5N1-HA
# structure unrelated to this peptide or allele) has been dropped rather
# than reassigned. No crystal structure of RPRGEVRFL bound to any HLA
# allele could be found on RCSB, PDBe, or PDBj after a thorough search.
# The benchmark set is therefore 4 structures (B1, B2, B4, B5), not 5.
# If a genuine RPRGEVRFL structure is later confirmed to exist, it should
# be reinstated as a 5th independent benchmark rather than reusing 3VCL,
# since 3VCL is already serving as the B4 reference.

for p in BENCHMARK_PAIRS:
    print(f"{p.pair_id}: {p.hla_allele} / {p.peptide} (PDB {p.pdb_id}) — {p.note}")

In [ ]:
# Extracellular-domain sequences for AF3 input construction
# Class I: alpha chain (HLA heavy chain) + beta-2-microglobulin (invariant) + peptide
# Class II: DRA (invariant alpha) + DRB1 beta chain (variable) + peptide

HLA_SEQUENCES = {
    "HLA-A*02:01": "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*07:02": "GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPRGEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSELDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-DRB1*01:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEFRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
}

DRA_SEQ = "MAISGVPVLGFFIIAVLMSAQESWAIKEEHVIIQAEFYLNPDQSGEFMFDFDGDEIFHVDMAKKETVWRLEEFGRFASFEAQGALANIAVDKANLEIMTKRSNYTPITNVPPEVTVLTNSPVELREPNVLICFIDKFTPPVVNVTWLRNGKPVTTGVSETVFLPREDHLFRKFHYLPFLPSTEDVYDCRVEHWGLDEPLLKHWEFDAPSPLPETTENVVCALGLTVGLVGIIIGTIFIKQKGSSS"
B2M_SEQ  = "MSRSVALAVLALLSLSGLEAIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"

def is_class_ii(allele: str) -> bool:
    return "DRB1" in allele

print("Sequence dictionary loaded:", list(HLA_SEQUENCES.keys()))

In [ ]:
def build_af3_job(pair: BenchmarkPair) -> dict:
    """
    Build one AlphaFold Server dialect job (a single dict; the server accepts
    a JSON list of these). Class I: HLA heavy chain (A) + B2M (B) + peptide (C).
    Class II: DRA (A) + DRB1 beta (B) + peptide (C).
    """
    hla_seq = HLA_SEQUENCES.get(pair.hla_allele)
    if hla_seq is None:
        raise ValueError(f"No sequence stored for {pair.hla_allele}")

    if is_class_ii(pair.hla_allele):
        sequences = [
            {"proteinChain": {"sequence": DRA_SEQ, "count": 1}},
            {"proteinChain": {"sequence": hla_seq, "count": 1}},
            {"proteinChain": {"sequence": pair.peptide, "count": 1}},
        ]
    else:
        sequences = [
            {"proteinChain": {"sequence": hla_seq, "count": 1}},
            {"proteinChain": {"sequence": B2M_SEQ, "count": 1}},
            {"proteinChain": {"sequence": pair.peptide, "count": 1}},
        ]

    return {
        "name": f"MS3_step3_{pair.pair_id}_{pair.pdb_id}",
        "modelSeeds": [1],
        "sequences": sequences,
        "dialect": "alphafoldserver",
        "version": 1,
    }

# Generate one JSON file per pair + a manifest for tracking
manifest_rows = []
for pair in BENCHMARK_PAIRS:
    job = build_af3_job(pair)
    job_path = FASTA_DIR / f"{pair.pair_id}_{pair.pdb_id}_af3input.json"
    job_path.write_text(json.dumps([job], indent=2))  # AF3 server dialect = top-level LIST

    expected_result_dir = RESULTS_DIR / f"{pair.pair_id}_{pair.pdb_id}"
    manifest_rows.append({
        "pair_id": pair.pair_id,
        "hla_allele": pair.hla_allele,
        "peptide": pair.peptide,
        "pdb_id": pair.pdb_id,
        "af3_input_file": str(job_path),
        "expected_result_folder": str(expected_result_dir),
        "submitted": False,
        "downloaded": False,
    })
    print(f"Wrote {job_path.name}  ->  expected results in {expected_result_dir.name}/")

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = BASE_DIR / "submission_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved: {manifest_path}")
manifest_df

## [MANUAL STEP] Submit to AlphaFold3 Server

The free AF3 server (**alphafoldserver.com**) has no public batch/API endpoint — Google's
hosted version only accepts submissions through the browser. This is a hard constraint
of the free tier, not a gap in this notebook. Do the following for each of the 5 files
in `af3_inputs/`:

1. Go to **https://alphafoldserver.com** and sign in
2. Click "Upload JSON" (or paste the file contents) and submit the job
3. Wait for the job to complete (typically a few minutes; the server has a
   **~10-20 jobs/day** rate limit on free accounts — 5 jobs for this step is well within that)
4. Download the result (AF3 provides a `.zip` containing `*_model.cif` and a
   `*_confidences.json` per job)
5. Unzip it into the **exact folder name** shown in the `expected_result_folder`
   column above, e.g. `af3_results/B1_1OGA/`

Once all 5 folders exist under `af3_results/`, re-run the cell below and continue
to Part B — the notebook will find them automatically and tell you if any are missing.

In [ ]:
# Check which results have arrived
def check_results():
    status = []
    for _, row in manifest_df.iterrows():
        folder = Path(row["expected_result_folder"])
        cif_files = list(folder.glob("*.cif")) if folder.exists() else []
        conf_files = list(folder.glob("*confidence*.json")) if folder.exists() else []
        status.append({
            "pair_id": row["pair_id"],
            "folder_exists": folder.exists(),
            "structure_found": len(cif_files) > 0,
            "confidence_found": len(conf_files) > 0,
            "ready": folder.exists() and len(cif_files) > 0,
        })
    return pd.DataFrame(status)

status_df = check_results()
print(status_df.to_string(index=False))

n_ready = status_df["ready"].sum()
print(f"\n{n_ready}/{len(status_df)} pairs ready. ", end="")
if n_ready < len(status_df):
    print("Upload the missing result folders before continuing to Part B.")
else:
    print("All pairs ready — proceed to Part B.")

## Part B — Structural Benchmark (TM-score / RMSD vs. Crystal Structures)

Downloads each reference crystal structure from RCSB and compares it against the
AF3-predicted structure. Uses the `TMscore` binary if available on the runtime;
falls back to a BioPython Cα RMSD (reported honestly — TM-score is left blank
rather than approximated, since it is not a valid substitute).

In [ ]:
def download_reference_pdb(pdb_id: str) -> Path:
    dest = PDB_DIR / f"{pdb_id.lower()}.pdb"
    if dest.exists():
        return dest
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    dest.write_bytes(r.content)
    return dest

def find_af3_structure(pair_id: str, pdb_id: str) -> Optional[Path]:
    folder = RESULTS_DIR / f"{pair_id}_{pdb_id}"
    cifs = list(folder.glob("*.cif"))
    return cifs[0] if cifs else None

def biopython_rmsd(pred_path: Path, ref_path: Path) -> dict:
    """Superimpose Cα atoms; handles both .cif (AF3 output) and .pdb (crystal ref)."""
    from Bio.PDB import MMCIFParser, PDBParser, Superimposer

    def load_structure(path):
        parser = MMCIFParser(QUIET=True) if path.suffix == ".cif" else PDBParser(QUIET=True)
        return parser.get_structure(path.stem, str(path))

    try:
        pred = load_structure(pred_path)
        ref = load_structure(ref_path)
    except Exception as e:
        return {"rmsd_angstrom": None, "rmsd_method": f"parse_error: {e}"}

    pred_ca = [a for a in pred.get_atoms() if a.get_name() == "CA"]
    ref_ca = [a for a in ref.get_atoms() if a.get_name() == "CA"]
    n = min(len(pred_ca), len(ref_ca))
    if n == 0:
        return {"rmsd_angstrom": None, "rmsd_method": "no_ca_atoms"}

    sup = Superimposer()
    sup.set_atoms(ref_ca[:n], pred_ca[:n])
    return {"rmsd_angstrom": round(sup.rms, 3), "rmsd_method": "biopython_ca"}

def run_tmscore(pred_path: Path, ref_path: Path) -> dict:
    """Try the TMscore binary if present on PATH; otherwise fall back cleanly."""
    import subprocess, shutil
    if shutil.which("TMscore") and pred_path.suffix == ".pdb" and ref_path.suffix == ".pdb":
        try:
            result = subprocess.run(["TMscore", str(pred_path), str(ref_path)],
                                     capture_output=True, text=True, timeout=60)
            tm_score = None
            for line in result.stdout.splitlines():
                if line.startswith("TM-score="):
                    tm_score = float(line.split("=")[1].split()[0])
            if tm_score is not None:
                return {"tm_score": tm_score, **biopython_rmsd(pred_path, ref_path)}
        except Exception:
            pass
    # Fallback: BioPython RMSD only, TM-score left blank (not approximated)
    return {"tm_score": None, **biopython_rmsd(pred_path, ref_path)}

benchmark_rows = []
for pair in BENCHMARK_PAIRS:
    row = {"pair_id": pair.pair_id, "hla_allele": pair.hla_allele,
           "peptide": pair.peptide, "pdb_id": pair.pdb_id}

    af3_struct = find_af3_structure(pair.pair_id, pair.pdb_id)
    if af3_struct is None:
        print(f"  [SKIP] {pair.pair_id}: no AF3 structure found yet")
        row.update({"tm_score": None, "rmsd_angstrom": None, "rmsd_method": "no_af3_structure"})
        benchmark_rows.append(row)
        continue

    try:
        ref_pdb = download_reference_pdb(pair.pdb_id)
    except requests.RequestException as e:
        print(f"  [WARN] {pair.pair_id}: RCSB download failed: {e}")
        ref_pdb = None

    if ref_pdb:
        row.update(run_tmscore(af3_struct, ref_pdb))
    else:
        row.update({"tm_score": None, "rmsd_angstrom": None, "rmsd_method": "ref_download_failed"})

    benchmark_rows.append(row)
    print(f"  {pair.pair_id}: TM-score={row.get('tm_score')}  "
          f"RMSD={row.get('rmsd_angstrom')} A ({row.get('rmsd_method')})")

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df

## Part C — RCSB Metadata (Resolution, Deposit Year, Method)

Pulls the ground-truth structure's own quality metadata — reviewers reasonably
expect to see how recent/high-resolution the benchmark structures are.

In [ ]:
RCSB_ENTRY_API = "https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"

def fetch_rcsb_metadata(pdb_id: str) -> dict:
    try:
        r = requests.get(RCSB_ENTRY_API.format(pdb_id=pdb_id), timeout=20)
        r.raise_for_status()
        data = r.json()
    except (requests.RequestException, ValueError) as e:
        print(f"  [WARN] RCSB metadata fetch failed for {pdb_id}: {e}")
        return {"resolution_a": None, "deposit_year": None, "exp_method": None}

    res_list = data.get("rcsb_entry_info", {}).get("resolution_combined")
    resolution = res_list[0] if res_list else None

    deposit_date = data.get("rcsb_accession_info", {}).get("deposit_date")
    deposit_year = deposit_date[:4] if deposit_date else None

    methods = data.get("exptl")
    exp_method = methods[0].get("method") if methods else None

    return {"resolution_a": resolution, "deposit_year": deposit_year, "exp_method": exp_method}

meta_rows = []
for pair in BENCHMARK_PAIRS:
    meta = fetch_rcsb_metadata(pair.pdb_id)
    meta_rows.append({"pair_id": pair.pair_id, "pdb_id": pair.pdb_id, **meta})
    print(f"  {pair.pdb_id}: {meta}")

meta_df = pd.DataFrame(meta_rows)
benchmark_df = benchmark_df.merge(meta_df, on=["pair_id", "pdb_id"], how="left")
benchmark_df

## Part D — NetMHCpan / NetMHCIIpan Concordance

NetMHCpan 4.1 (class I) and NetMHCIIpan 4.0 (class II) require a free academic license
and a local install — they cannot run inside Colab directly. Run the commands below on
your own machine, then upload the resulting `.xls`/`.txt` output back into this notebook.

```bash
# On your local machine, per peptide (class I example):
netMHCpan -a HLA-A02:01 -p peptide.txt -BA -xls -xlsfile B1_netmhcpan.xls

# Class II example:
netMHCIIpan -a DRB1_0101 -inptype 1 -f peptide.txt -BA -xls -xlsfile B5_netmhciipan.xls
```

Upload each result file below (or skip this section and re-run later once you have them).

In [ ]:
NETMHC_DIR = BASE_DIR / "netmhc_results"
NETMHC_DIR.mkdir(exist_ok=True)

# Colab file upload widget — skip this cell if running outside Colab or if you
# would rather manually copy files into NETMHC_DIR
try:
    from google.colab import files
    print("Upload your NetMHCpan/NetMHCIIpan output files now (or press Cancel to skip):")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (NETMHC_DIR / fname).write_bytes(content)
        print(f"  Saved: {fname}")
except ImportError:
    print("Not running in Colab — place NetMHCpan output files manually into:", NETMHC_DIR)

In [ ]:
def parse_netmhcpan_xls(path: Path) -> Optional[dict]:
    """
    Parses standard NetMHCpan/NetMHCIIpan -xls tabular output.
    Returns the top-ranked row's %Rank_EL, affinity (nM), and bind level.
    Column layout varies slightly between NetMHCpan and NetMHCIIpan versions,
    so this looks for the standard column names rather than fixed positions.
    """
    try:
        df = pd.read_csv(path, sep="\t", skiprows=1)
    except Exception as e:
        print(f"  [WARN] Could not parse {path.name}: {e}")
        return None

    rank_col = next((c for c in df.columns if "Rank" in c and "EL" in c), None)
    aff_col = next((c for c in df.columns if "nM" in c or "Aff" in c), None)
    bind_col = next((c for c in df.columns if "Bind" in c), None)

    if rank_col is None:
        print(f"  [WARN] Could not find %Rank_EL column in {path.name}")
        return None

    top = df.iloc[0]
    return {
        "netmhc_rank_el": top.get(rank_col),
        "netmhc_ic50_nm": top.get(aff_col) if aff_col else None,
        "netmhc_bind_level": top.get(bind_col) if bind_col else None,
    }

netmhc_rows = []
for pair in BENCHMARK_PAIRS:
    matches = list(NETMHC_DIR.glob(f"{pair.pair_id}_*"))
    if not matches:
        netmhc_rows.append({"pair_id": pair.pair_id, "netmhc_rank_el": None,
                             "netmhc_ic50_nm": None, "netmhc_bind_level": None})
        continue
    result = parse_netmhcpan_xls(matches[0])
    netmhc_rows.append({"pair_id": pair.pair_id, **(result or {
        "netmhc_rank_el": None, "netmhc_ic50_nm": None, "netmhc_bind_level": None})})

netmhc_df = pd.DataFrame(netmhc_rows)
benchmark_df = benchmark_df.merge(netmhc_df, on="pair_id", how="left")
benchmark_df

## Part E — Export Manuscript Table

Saves the combined benchmark table (structural accuracy + NetMHCpan concordance)
for direct use in the manuscript.

In [ ]:
FINAL_COLUMNS = [
    "pair_id", "hla_allele", "peptide", "pdb_id",
    "resolution_a", "deposit_year", "exp_method",
    "tm_score", "rmsd_angstrom", "rmsd_method",
    "netmhc_rank_el", "netmhc_ic50_nm", "netmhc_bind_level",
]
final_df = benchmark_df.reindex(columns=FINAL_COLUMNS)

out_path = TABLES_DIR / "Table_S0_af3_benchmark.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}\n")
final_df

In [ ]:
# Download the final table to your local machine
try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    print(f"Table saved locally at: {out_path}")

n_benchmarked = final_df["tm_score"].notna().sum() + (final_df["tm_score"].isna() & final_df["rmsd_angstrom"].notna()).sum()
print(f"\nStep 3 complete: {n_benchmarked}/{len(final_df)} pairs structurally compared.")
print("Feed this table into the manuscript as the AF3-vs-crystal-structure benchmark evidence,")
print("then proceed to Step 4 (full 30-40 pair panel).")